In [1]:
import warnings
import pandas as pd
warnings.filterwarnings("ignore")

from building_models.commons_functions.parsers_commons import ParsersCommons
from building_models.utils.constants import COLUMNS_TO_WORK
from building_models.utils.utils_functions import UtilsFunctions

#### Antioxidant protein dataset preprocessing and metadata generation

This script processes antioxidant protein data from a single source file, extracts labels from sequence identifiers, checks data consistency, and generates a structured dataset along with metadata for downstream classification tasks.

- Overview
    - Task: Antioxidant protein classification dataset preparation
    - Source: ANOX dataset
    - Input: FASTA-like file containing protein sequences with embedded labels and metadata Excel file
    - Output: Processed dataset (CSV) and metadata file (JSON)
- Process:
    - Read protein sequences from the input file
    - Extract labels from sequence identifiers
    - Remove identifier column after label extraction
    - Check for duplicated sequences and label consistency
    - Load and organize source metadata
    - Export data

- Auxiliary variables

In [2]:
path_export = "../../processed_dataset"
path_input = "../../raw_dataset"
metadata_file = "../../raw_dataset/raw_data_description.xlsx"
name_task = "antioxidant_classification"
name_source = "ANOX"

- Read doc

In [3]:
df_data = ParsersCommons.read_fasta_doc(f"{path_input}/{name_source}/anti_protein_positive_negative.txt")
df_data["label"] = df_data["id"].str.split("|").str[-1]
df_data.head()

,id,sequence,label
0,anti_|1,MTKGILLGDKFPDFRAETNEGFIPSFYDWIGKDSWAILFSHPRDFT...,1
1,anti_|1,MLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMN...,1
2,anti_|1,MAIALSSSSTITSITLQPKLKTIHGLGTVLPGYSVKSHFRSVSLRR...,1
3,anti_|1,MITSSKKIVSAMLSTSLWIGVASAAYAETTNVEAEGYSTIGGTYQD...,1
4,anti_|1,MANSGLWELITIGSAVRNVAKSYLKAEASSITAKQLYDASKITSSK...,1


In [4]:
df_data = df_data.drop(columns=["id"])

- Checking duplicates

In [5]:
df_consistent_duplicates, df_errors, df_unique = ParsersCommons.processing_duplicated(
    df_data, group_seq= "sequence",
    label_col= "label")
df_consistent_duplicates.shape, df_errors.shape, df_unique.shape

((0, 0), (0, 0), (1805, 2))

In [6]:
df_data["label"].value_counts()

label
0    1552
1     253
Name: count, dtype: int64

- Working with metadata

In [7]:
metadata_file = ParsersCommons.read_metadata(metadata_file, name_source=name_source, columns_to_select=COLUMNS_TO_WORK)
metadata_file.head()

,name dataset,name source,type source,static-dynamic,license,reports constant updates,year of publication,last update date,download date,file format,protein format,category dataset,task,obtaining negative dataset,obtaining positive dataset,repository or server,publication,unit of measurement
3,anti_protein_positive_negative.txt,ANOX,Dataset,Static,No information,No,2021,2021-10-15,2025-09-07,txt,"Sequence, UniProt ID",Enzyme/protein classification,Antioxidant,Previously published model dataset,Previously published model dataset,https://github.com/NWAFU-LiuLab/ANOX/tree/main...,https://www.sciencedirect.com/science/article/...,No information


In [8]:
dict_metadata = ParsersCommons.create_metadata_from_file(metadata_file)
dict_metadata

{'name dataset': 'anti_protein_positive_negative.txt',
 'name source': 'ANOX',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'reports constant updates': 'No',
 'year of publication': 2021,
 'last update date': Timestamp('2021-10-15 00:00:00'),
 'download date': Timestamp('2025-09-07 00:00:00'),
 'file format': 'txt',
 'protein format': 'Sequence, UniProt ID',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'Previously published model dataset',
 'obtaining positive dataset': 'Previously published model dataset',
 'repository or server': 'https://github.com/NWAFU-LiuLab/ANOX/tree/main/Benchmark%20dataset',
 'publication': 'https://www.sciencedirect.com/science/article/pii/S0003269721001585?via%3Dihub',
 'unit of measurement': 'No information',
 'number_of_sources': 1,
 'processing_date': '2026-08-01 19:42:24'}

In [9]:
df_data["label"] = df_data["label"].astype(int)

In [10]:
dict_metadata['number_of_records'] = df_data.shape[0]
dict_metadata['number_of_collected_sequences'] = df_data.shape[0]
dict_metadata['number_of_unique_sequences'] = df_data.shape[0]
dict_metadata['positive_examples'] = df_data[df_data["label"] == 1].shape[0]
dict_metadata['negative_examples'] = df_data[df_data["label"] == 0].shape[0]
dict_metadata['number_of_sequences_with_errors'] = df_errors.shape[0]
dict_metadata

{'name dataset': 'anti_protein_positive_negative.txt',
 'name source': 'ANOX',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'reports constant updates': 'No',
 'year of publication': 2021,
 'last update date': Timestamp('2021-10-15 00:00:00'),
 'download date': Timestamp('2025-09-07 00:00:00'),
 'file format': 'txt',
 'protein format': 'Sequence, UniProt ID',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'Previously published model dataset',
 'obtaining positive dataset': 'Previously published model dataset',
 'repository or server': 'https://github.com/NWAFU-LiuLab/ANOX/tree/main/Benchmark%20dataset',
 'publication': 'https://www.sciencedirect.com/science/article/pii/S0003269721001585?via%3Dihub',
 'unit of measurement': 'No information',
 'number_of_sources': 1,
 'processing_date': '2026-08-01 19:42:24',
 'number_of_records': 1805,
 'number_of_collected_sequences': 1805,
 'number_o

- Export data

In [11]:
UtilsFunctions.make_directory(f"{path_export}/{name_task}/{name_source}")
UtilsFunctions.export_json(f"{path_export}/{name_task}/{name_source}/metadata_{name_source}.json", dict_metadata)
df_data.to_csv(f"{path_export}/{name_task}/{name_source}/processed_data.csv", index=False)